# Data cleaning notebook

This notebook is the walkthrough of how we rescued the messy Track-1 UPI files.

We already have Python scripts under `Data/raw/src/cleaning/`. This notebook is
for an evaluator who wants to **see the decisions**, not just the output CSVs.

**Rule we followed:** fix the value if we can read it. Do not drop a row just
because the date is unix or the amount has `Rs.`. We only drop exact duplicate
rows after standardising.

| file | what was wrong |
| --- | --- |
| `track1_upi_transactions.csv` | unix + DD-MM-YY timestamps, `Rs.` amounts, status `F`/`S`/`COMPLETED` |
| `track1_merchants_master.csv` | `MCH 2430`, city `Hyd`/`Madras`, status `A`/`Live`, category slang |
| `track1_kyc_records.csv` | `USR 45454`, `₹11,214`, `27.3k` income, KYC `Done`/`V`, city `Bombay` |
| `track1_chargebacks.json` | merchant `3835`, reason `ato`/`no service`, severity `P1`/`H` |


## 0. Setup

Paths are relative to the repo root. Pandas is enough for this notebook.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# repo root = parent of notebooks/
ROOT = Path.cwd()
if not (ROOT / "Data" / "raw").exists():
    # if the kernel was started inside notebooks/
    ROOT = Path.cwd().parent

RAW = ROOT / "Data" / "raw"
PROC = ROOT / "Data" / "processed"

print("repo:", ROOT)
print("raw exists:", RAW.exists())
print("processed exists:", PROC.exists())
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.width", 120)


## 1. Load raw files and print row counts

This is the **before** number. Cleaning proof is raw vs cleaned, not "we
deleted whatever we didn't like".


In [ ]:
txn_raw = pd.read_csv(RAW / "track1_upi_transactions.csv", dtype=str)
mch_raw = pd.read_csv(RAW / "track1_merchants_master.csv", dtype=str)
kyc_raw = pd.read_csv(RAW / "track1_kyc_records.csv", dtype=str)

with open(RAW / "track1_chargebacks.json", encoding="utf-8") as f:
    cb_raw = pd.json_normalize(json.load(f)).astype(str)

raw_counts = pd.DataFrame({
    "table": ["upi_transactions", "merchants_master", "kyc_records", "chargebacks"],
    "raw_rows": [len(txn_raw), len(mch_raw), len(kyc_raw), len(cb_raw)],
})
raw_counts


## 2. What is actually messy?

We profiled missing values, exact duplicates, and mixed formats **before**
touching a row. If 50% of a column is blank we still keep the row — a
transaction with no MCC is still a payment.


In [ ]:
def profile(df, name):
    print(f"\n=== {name}  rows={len(df)}  cols={df.shape[1]} ===")
    print("exact duplicate rows:", int(df.duplicated().sum()))
    miss = df.isna().mean().sort_values(ascending=False).head(6)
    # empty strings also count as missing in this dump
    empty = (df.apply(lambda s: s.astype(str).str.strip().isin(["", "nan", "None"]))).mean()
    print("top empty-ish columns:")
    print((empty.sort_values(ascending=False).head(6) * 100).round(1).astype(str) + "%")

profile(txn_raw, "transactions")
profile(mch_raw, "merchants")
profile(kyc_raw, "kyc")
profile(cb_raw, "chargebacks")


## 3. Why we did not drop 50% of the rows

Dropping every "ugly" cell would have thrown away most of the dataset and
broken joins.

| problem | lazy option | what we did |
| --- | --- | --- |
| amount = `Rs. 1720.78` | drop row | strip `Rs.` / `₹` / `INR` / commas |
| timestamp = `1769823227` | drop row | parse unix **and** day-first strings |
| status = `F` / `COMPLETED` | drop row | map aliases → Success/Failed/Pending/Processing |
| city = `Bombay` / `Hyd` | drop row | city map (Bombay→Mumbai, Madras→Chennai, …) |
| merchant_id = `3835` | drop row | prefix `MCH` |
| income = `27.3k` | drop row | `k` suffix × 1000 |
| negative ticket size | drop merchant | set ticket to NA, **keep** the merchant |
| missing MCC | drop txn | fill from the other table if that merchant has one MCC |
| same user_id, two KYC rows | drop both | keep both in cleaned KYC, later pick the more complete row for `dim_customer` |

Only **exact duplicate rows** after standardising are removed. That is ~1–3%
per file, not 50%.


## 4. Standardisation examples (same txn_id / complaint_id)

These rows are copied from the raw files. After the cleaning scripts run,
the matching cleaned row should be parseable dates + numeric INR + one status
word.


In [ ]:
# --- transactions: currency, unix time, status slang ---
txn_ids = ["TXN00003140", "TXN00008060", "TXN00004294", "TXN00005067"]
print("RAW transactions")
display(txn_raw.loc[txn_raw["txn_id"].isin(txn_ids),
                    ["txn_id", "timestamp", "amount", "utr", "status"]])

# --- merchants: spaced ids, city nicknames, status letters ---
print("RAW merchants")
display(mch_raw.loc[
    mch_raw["merchant_id"].isin(["MCH 2430", "MCH4859", "mch4323"]),
    ["merchant_id", "merchant_category", "city", "merchant_status", "declared_avg_ticket_size"],
])

# --- kyc: income units + KYC slang + Bombay ---
print("RAW kyc")
display(kyc_raw.loc[
    kyc_raw["user_id"].astype(str).str.replace(" ", "").isin(["USR16112", "USR45454", "USR46189"]),
    ["user_id", "full_name", "monthly_income", "kyc_status", "city"],
].head(6))

# --- chargebacks: bare merchant id, Rs amount, P4/H severity ---
print("RAW chargebacks")
display(cb_raw.loc[
    cb_raw["complaint_id"].isin(["CBK0001941", "CBK0001799", "CBK0002663"]),
    ["complaint_id", "merchant_id", "disputed_amount", "reason_code", "severity"],
])


## 5. The actual cleaning (run the scripts)

The logic lives in:

- `Data/raw/src/cleaning/clean upi_trans and merchants_master.py`
- `Data/raw/src/cleaning/clean kyc_records.py`
- `Data/raw/src/cleaning/clean chargebacks.py`

Each script prints `raw rows` → `cleaned rows` when it finishes.

If `Data/processed/` already has the cleaned CSVs you can skip this cell.
Re-running is safe (it overwrites the same files).


In [ ]:
import runpy

CLEAN = RAW / "src" / "cleaning"

# Uncomment the next three lines to rebuild cleaned_* from raw.
# runpy.run_path(str(CLEAN / "clean upi_trans and merchants_master.py"), run_name="__main__")
# runpy.run_path(str(CLEAN / "clean kyc_records.py"), run_name="__main__")
# runpy.run_path(str(CLEAN / "clean chargebacks.py"), run_name="__main__")

print("Using existing files in", PROC)
print("To rebuild, uncomment the runpy lines above and re-run this cell.")


## 6. Proof: raw row count vs cleaned row count

This is the same table `prove_row_counts.py` prints. **Kept ≈ 98%.**


In [ ]:
txn_cln = pd.read_csv(PROC / "cleaned_upi_transactions.csv", dtype=str)
mch_cln = pd.read_csv(PROC / "cleaned_merchants_master.csv", dtype=str)
kyc_cln = pd.read_csv(PROC / "cleaned_kyc_records.csv", dtype=str)
cb_cln = pd.read_csv(PROC / "cleaned_chargebacks.csv", dtype=str)

proof = pd.DataFrame({
    "table": ["upi_transactions", "merchants_master", "kyc_records", "chargebacks"],
    "raw_rows": [len(txn_raw), len(mch_raw), len(kyc_raw), len(cb_raw)],
    "cleaned_rows": [len(txn_cln), len(mch_cln), len(kyc_cln), len(cb_cln)],
})
proof["dropped"] = proof["raw_rows"] - proof["cleaned_rows"]
proof["kept_pct"] = (proof["cleaned_rows"] / proof["raw_rows"] * 100).round(1)
proof.loc[len(proof)] = [
    "TOTAL",
    proof["raw_rows"].sum(),
    proof["cleaned_rows"].sum(),
    proof["dropped"].sum(),
    round(proof["cleaned_rows"].sum() / proof["raw_rows"].sum() * 100, 1),
]
proof


## 7. Same rows after cleaning

If these still look like `Rs.` or unix seconds, the scripts did not run.


In [ ]:
print("CLEANED transactions")
display(txn_cln.loc[txn_cln["txn_id"].isin(txn_ids),
                    ["txn_id", "timestamp", "amount", "utr", "status"]])

print("CLEANED merchants")
display(mch_cln.loc[
    mch_cln["merchant_id"].isin(["MCH2430", "MCH4859", "MCH4323"]),
    ["merchant_id", "merchant_category", "city", "merchant_status", "declared_avg_ticket_size"],
].drop_duplicates("merchant_id"))

print("CLEANED kyc")
display(kyc_cln.loc[
    kyc_cln["user_id"].isin(["USR16112", "USR45454", "USR46189"]),
    ["user_id", "full_name", "monthly_income", "kyc_status", "city"],
].drop_duplicates("user_id"))

print("CLEANED chargebacks")
display(cb_cln.loc[
    cb_cln["complaint_id"].isin(["CBK0001941", "CBK0001799", "CBK0002663"]),
    ["complaint_id", "merchant_id", "disputed_amount", "reason_code", "severity"],
])


## 8. Missing values — impute, don't delete

**MCC:** a blank MCC on a transaction is filled from `merchants_master` only
when that merchant has **one** MCC in the master. If two MCCs disagree we
leave it blank (guessing would invent a category).

**Negative ticket size:** compared against the merchant's real average
payment. It was not a simple sign flip, so we null the field and keep the
merchant. Deleting the merchant would orphan their transactions.

**Duplicate user_id in KYC:** not exact-row dups. `create_unique_customers.py`
scores completeness + Approved KYC and keeps the best profile for
`dim_customer`. The extra KYC rows are not thrown away until that step.


In [ ]:
# MCC still missing after clean (we refused to guess)
print("txn MCC nulls after clean:", txn_cln["mcc"].isna().sum())

# dim tables: one row per id (best record kept, not random drop)
dim_c = pd.read_csv(PROC / "dim_customer.csv", dtype=str)
dim_m = pd.read_csv(PROC / "dim_merchant.csv", dtype=str)
print("cleaned kyc rows:", len(kyc_cln), " unique user_id:", kyc_cln["user_id"].nunique())
print("dim_customer rows:", len(dim_c), " unique user_id:", dim_c["user_id"].nunique())
print("cleaned merchants:", len(mch_cln), " unique merchant_id:", mch_cln["merchant_id"].nunique())
print("dim_merchant rows:", len(dim_m), " unique merchant_id:", dim_m["merchant_id"].nunique())


## 9. Relationships are not ignored

`build_fact_tables.py` left-joins facts to dimensions and writes match flags.

We **keep unmatched ids**. A payment whose `user_id` is not in KYC is still
a payment (possible mule / orphan). A hard FOREIGN KEY would have deleted it.


In [ ]:
ft = pd.read_csv(PROC / "fact_transactions.csv", dtype=str)
fc = pd.read_csv(PROC / "fact_chargebacks.csv", dtype=str)

print("fact_transactions:", len(ft))
print(ft["customer_match_status"].value_counts(dropna=False))
print(ft["merchant_match_status"].value_counts(dropna=False))
print()
print("fact_chargebacks:", len(fc))
print("transaction_match_status:")
print(fc["transaction_match_status"].value_counts(dropna=False))
print("user_consistency_status:")
print(fc["user_consistency_status"].value_counts(dropna=False))


## 10. Re-run everything from one command

From the repo root (needs pandas):

```bash
pip install -r requirements.txt
python Data/raw/src/run_pipeline.py
python Data/raw/src/cleaning/prove_row_counts.py
```

Open this notebook:

```bash
jupyter notebook notebooks/01_data_cleaning.ipynb
```

Column meanings for the warehouse tables are in the README **Data dictionary**.
